# Megaline Revenue Analysis

## Introdução

Neste projeto, vou analisar dados da empresa de telecomunicações Megaline, que oferece dois planos pré-pagos: Surf e Ultimate.

O objetivo é entender o comportamento dos clientes, calcular a receita mensal gerada por cada usuário e verificar qual plano gera mais receita em média. Também serão realizados testes estatísticos para comparar as receitas entre os planos e entre usuários da região NY-NJ e de outras regiões.

## 1. Inicialização

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## 2. Carregue os dados

In [ ]:
calls = pd.read_csv('/datasets/megaline_calls.csv')
internet = pd.read_csv('/datasets/megaline_internet.csv')
messages = pd.read_csv('/datasets/megaline_messages.csv')
plans = pd.read_csv('/datasets/megaline_plans.csv')
users = pd.read_csv('/datasets/megaline_users.csv')

## 3. Prepare os dados

Primeiro, vamos observar as informações gerais e as primeiras linhas de cada tabela.

In [ ]:
for name, df in {
    'calls': calls,
    'internet': internet,
    'messages': messages,
    'plans': plans,
    'users': users
}.items():
    print('\n---', name, '---')
    display(df.head())
    df.info()

### Conclusão inicial

Os dados estão divididos em cinco tabelas. As tabelas de chamadas, internet e mensagens possuem datas em formato de texto, então será necessário converter essas colunas para o tipo `datetime`.

A tabela `users` possui valores ausentes em `churn_date`, mas isso é esperado, pois significa que o cliente ainda estava ativo quando os dados foram extraídos. Não foram encontrados valores ausentes relevantes nas tabelas de uso.

## 4. Corrija os dados

In [ ]:
calls['call_date'] = pd.to_datetime(calls['call_date'])
internet['session_date'] = pd.to_datetime(internet['session_date'])
messages['message_date'] = pd.to_datetime(messages['message_date'])
users['reg_date'] = pd.to_datetime(users['reg_date'])
users['churn_date'] = pd.to_datetime(users['churn_date'])

## 5. Enriqueça os dados

Vamos criar a coluna de mês e arredondar cada chamada para cima, conforme a regra da Megaline.

In [ ]:
calls['month'] = calls['call_date'].dt.month
internet['month'] = internet['session_date'].dt.month
messages['month'] = messages['message_date'].dt.month

calls['duration_rounded'] = np.ceil(calls['duration']).astype(int)

### Comentário

A Megaline arredonda cada chamada individual para cima. Por isso, mesmo chamadas com poucos segundos são contadas como 1 minuto. Para internet, o arredondamento será feito depois, no total mensal de cada usuário.

## 6. Estude as condições dos planos

In [ ]:
plans

## 7. Agregue os dados por usuário e por mês

In [ ]:
calls_agg = calls.groupby(['user_id', 'month']).agg(
    calls=('id', 'count'),
    minutes=('duration_rounded', 'sum')
).reset_index()

messages_agg = messages.groupby(['user_id', 'month']).agg(
    messages=('id', 'count')
).reset_index()

internet_agg = internet.groupby(['user_id', 'month']).agg(
    mb_used=('mb_used', 'sum')
).reset_index()

internet_agg['gb_used'] = np.ceil(internet_agg['mb_used'] / 1024).astype(int)

In [ ]:
monthly = calls_agg.merge(messages_agg, on=['user_id', 'month'], how='outer')
monthly = monthly.merge(
    internet_agg[['user_id', 'month', 'mb_used', 'gb_used']],
    on=['user_id', 'month'],
    how='outer'
)

monthly[['calls', 'minutes', 'messages', 'mb_used', 'gb_used']] = monthly[
    ['calls', 'minutes', 'messages', 'mb_used', 'gb_used']
].fillna(0)

monthly = monthly.merge(users[['user_id', 'city', 'plan']], on='user_id', how='left')
monthly = monthly.merge(plans, left_on='plan', right_on='plan_name', how='left')

monthly.head()

## 8. Calcule a receita mensal

In [ ]:
def calculate_revenue(row):
    extra_minutes = max(row['minutes'] - row['minutes_included'], 0) * row['usd_per_minute']
    extra_messages = max(row['messages'] - row['messages_included'], 0) * row['usd_per_message']
    extra_gb = max(row['gb_used'] - (row['mb_per_month_included'] / 1024), 0) * row['usd_per_gb']

    return row['usd_monthly_pay'] + extra_minutes + extra_messages + extra_gb

monthly['revenue'] = monthly.apply(calculate_revenue, axis=1)

monthly[['user_id', 'month', 'plan', 'minutes', 'messages', 'gb_used', 'revenue']].head()

## 9. Estude o comportamento do usuário

### 9.1 Chamadas

In [ ]:
avg_minutes_by_month = monthly.groupby(['month', 'plan'])['minutes'].mean().unstack()

avg_minutes_by_month.plot(kind='bar', figsize=(10, 5))
plt.title('Average Monthly Call Minutes by Plan')
plt.xlabel('Month')
plt.ylabel('Average minutes')
plt.legend(title='Plan')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

for plan in monthly['plan'].unique():
    monthly.loc[monthly['plan'] == plan, 'minutes'].plot(
        kind='hist',
        bins=30,
        alpha=0.5,
        label=plan
    )

plt.title('Monthly Minutes Usage Distribution')
plt.xlabel('Minutes per month')
plt.ylabel('Frequency')
plt.legend()
plt.show()

In [ ]:
monthly.groupby('plan')['minutes'].agg(['mean', 'var', 'std']).round(2)

In [ ]:
monthly.boxplot(column='minutes', by='plan', figsize=(8, 5))
plt.title('Monthly Minutes Distribution by Plan')
plt.suptitle('')
plt.xlabel('Plan')
plt.ylabel('Minutes per month')
plt.show()

### Conclusão sobre chamadas

O consumo médio de minutos é parecido entre os dois planos. Os usuários do Surf e do Ultimate usam, em média, cerca de 430 minutos por mês. Isso indica que o comportamento em chamadas não é muito diferente entre os planos.

### 9.2 Mensagens

In [ ]:
monthly.groupby('plan')['messages'].agg(['mean', 'var', 'std']).round(2)

In [ ]:
plt.figure(figsize=(10, 5))

for plan in monthly['plan'].unique():
    monthly.loc[monthly['plan'] == plan, 'messages'].plot(
        kind='hist',
        bins=30,
        alpha=0.5,
        label=plan
    )

plt.title('Monthly Messages Distribution')
plt.xlabel('Messages per month')
plt.ylabel('Frequency')
plt.legend()
plt.show()

### Conclusão sobre mensagens

Os usuários do plano Ultimate enviam, em média, mais mensagens que os usuários do plano Surf. Mesmo assim, a maioria dos usuários não chega perto do limite de mensagens do plano Ultimate.

### 9.3 Internet

In [ ]:
monthly.groupby('plan')['gb_used'].agg(['mean', 'var', 'std']).round(2)

In [ ]:
plt.figure(figsize=(10, 5))

for plan in monthly['plan'].unique():
    monthly.loc[monthly['plan'] == plan, 'gb_used'].plot(
        kind='hist',
        bins=30,
        alpha=0.5,
        label=plan
    )

plt.title('Monthly Internet Usage Distribution')
plt.xlabel('GB used per month')
plt.ylabel('Frequency')
plt.legend()
plt.show()

### Conclusão sobre internet

O uso médio de internet é parecido entre os planos, mas os usuários do Surf ultrapassam com mais frequência o limite incluído no pacote, o que gera cobranças extras e aumenta a receita média desse plano.

## 10. Receita

In [ ]:
monthly.groupby('plan')['revenue'].agg(['count', 'mean', 'median', 'var', 'std', 'sum']).round(2)

In [ ]:
monthly.boxplot(column='revenue', by='plan', figsize=(8, 5))
plt.title('Monthly Revenue Distribution by Plan')
plt.suptitle('')
plt.xlabel('Plan')
plt.ylabel('Monthly revenue (USD)')
plt.show()

In [ ]:
avg_revenue_by_month = monthly.groupby(['month', 'plan'])['revenue'].mean().unstack()

avg_revenue_by_month.plot(kind='line', marker='o', figsize=(10, 5))
plt.title('Average Monthly Revenue by Plan Over Time')
plt.xlabel('Month')
plt.ylabel('Average revenue (USD)')
plt.legend(title='Plan')
plt.show()

### Conclusão sobre receita

A receita média mensal do plano Surf foi de aproximadamente **$60.71**, enquanto a do plano Ultimate foi de aproximadamente **$72.31**.

Embora o plano Ultimate tenha uma mensalidade maior, muitos usuários do Surf ultrapassam os limites do pacote, principalmente em dados de internet, o que aumenta a receita gerada por esse plano.

## 11. Teste de hipóteses estatísticas

### Hipótese 1

Queremos testar se a receita média dos usuários dos planos Ultimate e Surf são diferentes.

- H0: a receita média dos usuários dos planos Ultimate e Surf é igual.
- H1: a receita média dos usuários dos planos Ultimate e Surf é diferente.

Como estamos comparando médias de dois grupos independentes, usaremos o teste t para amostras independentes. Como as variâncias são diferentes, usaremos `equal_var=False`.

In [ ]:
surf_revenue = monthly.loc[monthly['plan'] == 'surf', 'revenue']
ultimate_revenue = monthly.loc[monthly['plan'] == 'ultimate', 'revenue']

alpha = 0.05

results = stats.ttest_ind(surf_revenue, ultimate_revenue, equal_var=False)

print('p-value:', results.pvalue)

if results.pvalue < alpha:
    print('Reject H0: the average revenues are different.')
else:
    print('Fail to reject H0: there is not enough evidence that the average revenues are different.')

### Conclusão da hipótese 1

Como o valor-p é menor que 0,05, rejeitamos a hipótese nula. Isso indica que há evidência estatística de que as receitas médias dos planos Surf e Ultimate são diferentes.

### Hipótese 2

Queremos testar se a receita média dos usuários da área de NY-NJ é diferente da receita média dos usuários de outras regiões.

- H0: a receita média dos usuários da área de NY-NJ é igual à dos usuários de outras regiões.
- H1: a receita média dos usuários da área de NY-NJ é diferente da dos usuários de outras regiões.

In [ ]:
monthly['is_ny_nj'] = monthly['city'].str.contains('NY-NJ', regex=False)

ny_nj_revenue = monthly.loc[monthly['is_ny_nj'], 'revenue']
other_revenue = monthly.loc[~monthly['is_ny_nj'], 'revenue']

alpha = 0.05

results = stats.ttest_ind(ny_nj_revenue, other_revenue, equal_var=False)

print('p-value:', results.pvalue)

if results.pvalue < alpha:
    print('Reject H0: the average revenues are different.')
else:
    print('Fail to reject H0: there is not enough evidence that the average revenues are different.')

### Conclusão da hipótese 2

Como o valor-p é menor que 0,05, rejeitamos a hipótese nula. Isso indica que há evidência estatística de que a receita média dos usuários da área de NY-NJ é diferente da receita média dos usuários das demais regiões.

## Conclusão geral

Neste projeto, foram analisados os dados da Megaline para comparar os planos Surf e Ultimate.

Durante a preparação dos dados, as colunas de datas foram convertidas para o formato correto, as chamadas foram arredondadas para cima conforme as regras da empresa e o consumo mensal de chamadas, mensagens e internet foi agregado por usuário.

A análise mostrou que o comportamento dos usuários em chamadas é parecido entre os planos. O uso de internet também é semelhante, mas os usuários do plano Surf ultrapassam mais frequentemente o limite incluído no pacote, gerando cobranças extras.

A receita média mensal do plano Surf foi de aproximadamente **$60.71**, enquanto a do plano Ultimate foi de aproximadamente **$72.31**. O teste estatístico confirmou que há diferença significativa entre as receitas médias dos dois planos.

Também foi encontrada diferença significativa entre a receita média dos usuários da região NY-NJ e dos usuários de outras regiões.

De forma geral, o plano Ultimate apresenta maior receita média mensal, mas o plano Surf também gera receitas relevantes devido às cobranças extras. Para decisões de marketing, a Megaline deve considerar tanto a receita média quanto o comportamento de exceder limites dos usuários do Surf.